# 1 · Classic constrained SOLA, starting with pixels

This is the first inspectable part of the proposed presentation. We keep the
original **unit box, 600 straight rays, smooth truth and noisy data**. We ask for
three fixed disk means (radius 0.12), including the original positive-anomaly disk.

We begin exactly as a classical discrete SOLA application would: parameterize
immediately by pixel values, form the forward matrix, use the **correct mass
matrix**, and solve constrained SOLA. Then refine only the numerical pixels.

The physical target disks, rays, noise covariance and trade-off stay fixed.
This tests numerical refinement; it does not shrink the requested averaging region.

In [ ]:
%matplotlib inline
from pathlib import Path
import hashlib, json, os, socket, subprocess, sys, time
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
import study as F
S=F.S
ex=F.Experiment(length_scale=.1)  # shared by both H2 routes; edit consistently in all notebooks
OUT=Path('output');OUT.mkdir(exist_ok=True)
COLORS=('#2563eb','#dc2626','#7c3aed')
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'grid.alpha':.2,'font.size':11})
print('host:',socket.gethostname(),'Python:',sys.executable)
print('source commit:',subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True).stdout.strip())
print('original problem SHA256:',hashlib.sha256(Path(S.P.__file__).read_bytes()).hexdigest())
print('600 original rays; one fixed noise draw; sigma =',S.P.SIGMA)
for label,c,t in zip(F.LABELS,ex.centres,ex.truth):print(label,c,'radius',ex.radius,'true disk mean',f'{t:.9f}')
print('Fixed normalized trade-off lambda =',ex.lam,'; H2 length scale =',ex.length_scale)
import platform, scipy, pygeoinf0 as pg
from threadpoolctl import threadpool_info
runtime=dict(host=socket.gethostname(),python=sys.version,numpy=np.__version__,scipy=scipy.__version__,
             platform=platform.platform(),cpu_count=os.cpu_count(),package_source=pg.__file__,
             threadpools=threadpool_info(),targets=ex.centres,radius=ex.radius,lambda_value=ex.lam,length_scale=ex.length_scale,
             study_sha256=hashlib.sha256(Path(F.__file__).read_bytes()).hexdigest(),
             helper_sha256=hashlib.sha256(Path(S.__file__).read_bytes()).hexdigest(),
             problem_sha256=hashlib.sha256(Path(S.P.__file__).read_bytes()).hexdigest())
F.save(OUT/'runtime.json',runtime)
print('BLAS thread counts:',[v['num_threads'] for v in runtime['threadpools']])


In [ ]:
z=np.linspace(0,1,220); xx,yy=np.meshgrid(z,z)
fig,ax=plt.subplots(1,2,figsize=(11,4),constrained_layout=True)
for a,b in zip(ex.p,ex.q):ax[0].plot(*np.column_stack([a,b]),color='.35',lw=.35,alpha=.15)
ax[0].scatter(*ex.src.T,s=18,marker='*',color='k');ax[0].scatter(*ex.stations.T,s=22,marker='^',color='k')
im=ax[1].imshow(S.P.true_model(xx,yy),origin='lower',extent=[0,1,0,1],cmap='RdBu_r')
for c,color,label in zip(ex.centres,COLORS,F.LABELS):
    for a in ax:a.add_patch(plt.Circle(c,ex.radius,fill=False,color=color,lw=2))
    ax[1].plot([],[],color=color,label=label)
for a in ax:a.set_aspect('equal');a.set_xlim(0,1);a.set_ylim(0,1);a.grid(False)
ax[0].set_title('The same observations throughout');ax[1].set_title('Known truth and fixed physical targets')
ax[1].legend(fontsize=8);fig.colorbar(im,ax=ax[1],shrink=.8);plt.show()


## Pixel coefficients are not an orthonormal basis

Let $m_h=\sum_j c_j\mathbf1_{C_j}$ on square pixels of side $h$.
The observation matrix contains exact ray lengths:
$$E_{ij}=\int_{\Gamma_i}\mathbf1_{C_j}\,ds.$$
The $L^2$ coefficient metric is $M_h=h^2I$, so
$$G_h^*=M_h^{-1}E^T,\qquad K_h=E M_h^{-1}E^T.$$
A transpose alone would be the adjoint in the **Euclidean coefficient metric**.
We retain the cell areas here. For the disk target density $t$, its coefficients
$t_h$ are exact cell averages from disk/cell overlap areas, and
$$b_h=E t_h.$$

The next cell verifies the weighted adjoint identity on an actual mesh.

In [ ]:
n=20;h=1/n
E=S.ray_matrix(ex.p,ex.q,n)
rng=np.random.default_rng(42);c=rng.normal(size=n*n);y=rng.normal(size=len(ex.d))
Gstar_y=np.asarray(E.T@y)/h**2
lhs=(E@c)@y;rhs=h**2*c@Gstar_y
print('data pairing:',lhs,'model pairing:',rhs,'difference:',abs(lhs-rhs))
assert np.allclose(lhs,rhs,rtol=2e-12,atol=2e-12)
print('Model dimension:',n*n,'Data dimension:',len(ex.d))
print('The huge diagonal mass matrix is applied by scaling, never stored densely.')


## The constrained SOLA solve

For each target, minimize
$$\frac{\|t-r_h\|_{L^2}^2}{\|t\|_{L^2}^2}
+\lambda\frac{x^TC_dx}{\sigma_{\rm ref}^2},\qquad v^Tx=1,$$
where $r_h=\sum_i x_i g_{i,h}$, $v_i=G_i1=L_i$,
$C_d=\sigma^2I$, $\sigma_{\rm ref}=\sigma$ and $\lambda=0.1$.
Thus the fixed matrix penalty is $\tau I$, with $\tau=\lambda\|t\|_{L^2}^2$.
We retain the **full physical target norm**; projecting the target adds only an
$x$-independent constant at this fixed penalty.

Set $H_h=K_h+\tau I$. The exact response correction is
$$x_0=H_h^{-1}b_h,\qquad
x=x_0+H_h^{-1}v\frac{1-v^Tx_0}{v^TH_h^{-1}v}.$$
There is one common Cholesky factor and four right-hand sides: three targets and
$v$. No weights are tuned on the realized data. The helper batches this formula.

In [ ]:
tau=ex.lam*ex.target_l2_norm2
system=F.pixel_system(ex,20)
K=system['K'];b=system['B'][0];v=ex.v
factor=la.cho_factor(K+tau*np.eye(len(v)),lower=True)
x0=la.cho_solve(factor,b);a=la.cho_solve(factor,v)
x=x0+a*(1-v@x0)/(v@a)
X=F.constrained_sola(K,system['B'],tau,v)
assert np.allclose(x,X[0],rtol=2e-12,atol=2e-13)
print('Physical constant responses:',X@v)
print('Estimates:',X@ex.d,'requested means:',ex.truth)


## Refine the pixels, not the experiment

We recompute $E_h$, its mass-corrected Gram and the target projections. We evaluate
**actual delivered truth** as $X_h d_{\rm clean}$ using accurately integrated
original rays. This avoids conflating a pixel approximation of the truth with
what the linear data transformation actually measures.

For every target,
$$X_hd-Tm=(X_hGm-Tm)+X_h\eta.$$
The standard deviation below describes $X_h\eta$, not the target discrepancy.

In [ ]:
rows=[];weights=[];pictures={}
max_n=int(os.environ.get('SOLA_PART1_MAX_PIXEL','2560'))
for n in F.PIXELS:
    if n>max_n:continue
    system=F.pixel_system(ex,n);X,row=F.solve_system(system,ex,tau)
    kernel2=np.einsum('ij,jk,ik->i',X,system['K'],X)
    mismatch2=ex.target_l2_norm2-2*np.sum(X*system['B'],axis=1)+kernel2
    row.update(n=n,h=1/n,dimension=n*n,kernel_norm=np.sqrt(kernel2).tolist(),
               relative_mismatch=np.sqrt(np.maximum(mismatch2,0)/ex.target_l2_norm2).tolist())
    rows.append(row);weights.append(X)
    if n in (20,80,320,max_n):
        pictures[n]=np.asarray(system['E'].T@X[0]).reshape(n,n)*n*n
    print(f"{n:4d}: estimates={np.round(row['estimate'],5)}, "
          f"separation ||x1-x2||={row['weight_separation']:.5g}, norm r1={row['kernel_norm'][0]:.5g}")
F.save(OUT/'pixels.json',dict(rows=rows,lambda_value=ex.lam,tau=tau))
np.savez_compressed(OUT/'pixel_weights.npz',weights=weights,grids=[r['n'] for r in rows])
assert max(np.max(np.abs(np.array(r['constant_response'])-1)) for r in rows)<2e-10


In [ ]:
ns=np.array([r['n'] for r in rows])
fig,ax=plt.subplots(1,3,figsize=(14,4),constrained_layout=True)
for k,(color,label) in enumerate(zip(COLORS,F.LABELS)):
    ax[0].semilogx(ns,[r['estimate'][k] for r in rows],'o-',color=color,label=label)
    ax[0].semilogx(ns,[r['delivered_truth'][k] for r in rows],':',color=color)
    ax[0].axhline(ex.truth[k],color=color,ls='--',alpha=.65)
    ax[1].loglog(ns,[r['std'][k] for r in rows],'o-',color=color)
    ax[2].semilogx(ns,[r['relative_mismatch'][k] for r in rows],'o-',color=color)
ax[0].legend(fontsize=8);ax[0].set_title('Estimate; dotted delivered truth; dashed target')
ax[1].set_title('Propagated noise decreases');ax[2].set_title('Relative L2 target mismatch')
for a in ax:a.set_xlabel('pixels per side (physical targets fixed)')
plt.show()
fig,ax=plt.subplots(1,2,figsize=(10,3.6),constrained_layout=True)
ax[0].loglog(ns,[r['weight_separation'] for r in rows],'o-');ax[0].set_ylabel(r'$\|x_h(q_1)-x_h(q_2)\|$')
for k,color in enumerate(COLORS):ax[1].loglog(ns,[r['kernel_norm'][k] for r in rows],'o-',color=color)
ax[1].set_ylabel(r'$\|r_h\|_{L^2}$ (can turn upward late)')
for a in ax:a.set_xlabel('pixels per side')
plt.show()


In [ ]:
fig,axes=plt.subplots(1,len(pictures),figsize=(14,3.7),constrained_layout=True)
lim=2*ex.target_l2_norm2
for ax,(n,density) in zip(np.atleast_1d(axes),pictures.items()):
    im=ax.imshow(density,origin='lower',extent=[0,1,0,1],cmap='RdBu_r',vmin=-lim,vmax=lim)
    ax.add_patch(plt.Circle(ex.centres[0],ex.radius,fill=False,color='k'))
    ax.set_title(f"{n} x {n}\npeak |density|={np.max(np.abs(density)):.1f}")
    ax.set_xticks([]);ax.set_yticks([]);ax.grid(False)
fig.colorbar(im,ax=np.atleast_1d(axes),shrink=.7,extend='both',label='cell density; common clipped colour scale')
plt.show()


## Physical questions and responses to a known moving anomaly

The top row shows the physical measures: an exact ray integral, a uniform disk
average, and the signed weighted rays selected by SOLA. A ray has zero area but
nonzero measure mass. The disk has density $1/(\pi R^2)$ inside its boundary.
The delivered measure has mass $\sum_i x_iL_i=1$ and retains its ray support.

For the bottom row, place the **same known unit-peak smooth test anomaly** at each
centre $z$ and ask what these physical measurements return:
$$G_iq_z,\qquad T_aq_z,\qquad (XG)_a q_z.$$
The test shape and its horizontal profile are shown explicitly. Its chosen
length is 0.1 throughout; it is neither a physical ray width nor information
about the unknown model. It has decaying tails, rather than compact support.
The heatmaps are **responses to test models**, not area-density representations
of the ray measures. Ray responses have units of model times length; target and
delivered averages have model units, with a shared colour scale for that pair.

This display probe does not alter the pixel/L2 matching metric or its SOLA weights.
We evaluate the actual transformed continuum-ray data for every test model,
rather than substituting a pixel approximation of that model.

In [ ]:
# physical-probe: pixels
import probe_views as V
probes=V.response_grid(ex,OUT)
V.plot_probe(ex,probes);plt.show()
V.plot_physical_and_responses(ex,probes,weights[-1][0],
    f"Pixel SOLA ({rows[-1]['n']} per axis): measures and physical responses");plt.show()
indices=[i for i,r in enumerate(rows) if r['n'] in {rows[0]['n'],80,320,rows[-1]['n']}]
V.plot_response_evolution(ex,probes,[weights[i][0] for i in indices],
    [f"{rows[i]['n']} x {rows[i]['n']} pixels" for i in indices],
    'Same test anomaly: how the actual pixel-SOLA questions change');plt.show()
print('All actual constant responses:',weights[-1]@ex.v)


## Why the correct mass matrix does not fix this continuum problem

Zero-width ray integration has no bounded extension to $L^2$ of area. The pixel
representers $g_{i,h}$ exist on each finite space; continuum $L^2$ representers
$g_i$ do not. For fixed generic rays, diagonal entries of $K_h$ grow like $1/h$.

Unit response prevents the zero functional, but does not preserve localization.
The density has unit mass on ray-touched cells of total area $O(h)$, forcing its
$L^2$ norm eventually to grow at least as $h^{-1/2}$. If
$K_h=a_h A+o(a_h)$ with $a_h\to\infty$ and $A>0$, bounded fixed target pairings
become negligible and
$$x_h(q)\longrightarrow\frac{A^{-1}v}{v^TA^{-1}v},$$
independent of target location. The common limit, if it exists, is a normalized
ray statistic. A displayed constant map would not establish a constant Earth.
Near-parallel geometry can delay this regime; the plotted meshes need not yet
reach it. No divergence of the data weights is asserted.

The transformed MDR remains valid. The pixel density's continuum volume-average
interpretation and the criterion's continuum limit are what need attention.
Next we fix a Hilbert formulation where the rays really are bounded.